# Transportation Problem

\begin{align*}
\text{Minimize} \quad & \sum_{i=1}^{m} \sum_{j=1}^{n} c_{ij}x_{ij} & \\
\text{Subject to} \quad & \sum_{j=1}^{n} x_{ij} \leq S_i & (i = 1, 2, \dots, m) \\
& \sum_{i=1}^{m} x_{ij} \geq D_j & (j = 1, 2, \dots, n) \\
& x_{ij} \geq 0 & (i = 1, 2, \dots, m,\ j = 1, 2, \dots, n)
\end{align*}

### Parameters
- $m$ and $n$ are the number of supply and demand locations respectively.
- $c_{ij}$ is the cost of shipping one unit of goods from supply location $i$ to demand location $j$.
- $S_i$ is the number of units of goods that can be shipped from supply location $i$.
- $D_j$ is the number of units of goods that need to be shipped to demand location $j$.

### Decision Variables
- $x_{ij}$: the number of units of goods shipped from supply location $i$ to demand location $j$.




###0. Preprocessing:
Set your Gurobi license, install `gurobipy`, and define the environment

In [1]:
!pip install gurobipy
import gurobipy as gp
from gurobipy import GRB, quicksum

# WLS Academic License — replace with your own credentials
# env = gp.Env(params={
#     'WLSACCESSID': '0649013d-2f94-464e-b68f-dc7d933b5822',
#     'WLSSECRET':   'a9386248-233e-42c3-9f08-56b178e9f9e2',
#     'LICENSEID':    0000000
# })

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 55.4 MB/s eta 0:00:00


###1. Import Libraries

In [2]:
import gurobipy as gp
from gurobipy import GRB, quicksum
import pandas as pd
import time

###2. Parameter Settings

Import the transportation data from the Excel file `Transportation.xlsx`.

In [3]:

df = pd.read_excel("Transportation.xlsx")

df.columns = [str(col).strip() for col in df.columns]


if "City" not in df.columns or "Capacity" not in df.columns:
    raise ValueError("Transportation.xlsx must contain 'City' and 'Capacity' columns.")


D = [col for col in df.columns if col not in ["City", "Capacity"]]

# Identify origin rows and demand row
city_clean = df["City"].astype(str).str.strip().str.lower()

origin_df = df[city_clean != "demand"].copy()
demand_df = df[city_clean == "demand"].copy()

if demand_df.empty:
    raise ValueError("No 'Demand' row found in the 'City' column.")

# Origin set O
O = origin_df["City"].astype(str).str.strip().tolist()

# Build cost parameter c[(i,j)]
c = {}
for _, row in origin_df.iterrows():
    i = str(row["City"]).strip()
    for j in D:
        c[i, j] = float(row[j])

# Build Supply and Demand dictionaries
Supply = {}
for _, row in origin_df.iterrows():
    i = str(row["City"]).strip()
    Supply[i] = float(row["Capacity"])

demand_row = demand_df.iloc[0]
Demand = {}
for j in D:
    Demand[j] = float(demand_row[j])

# Quick check
print("Origins (O):", O)
print("Destinations (D):", D)
print("Total Supply:", sum(Supply.values()))
print("Total Demand:", sum(Demand.values()))

Origins (O): ['Halifax', 'London', 'Ajax', 'Toronto', 'Montreal']
Destinations (D): ['Moncton', 'Sydney', 'Ottawa', 'Truro', 'Saint John', 'PEI', 'Hamilton', 'Waterloo']
Total Supply: 3000.0
Total Demand: 3000.0


In [4]:
# Separate origin rows from the Demand row
city_col = df['City'].astype(str).str.strip()
origin_df = df[city_col.str.lower() != 'demand'].copy()
demand_row = df[city_col.str.lower() == 'demand'].iloc[0]

O = origin_df['City'].astype(str).str.strip().tolist()           # Origins
D = [c for c in df.columns if c not in ['City', 'Capacity']]     # Destinations

In [5]:
c = {
    (str(row['City']).strip(), dest): float(row[dest])
    for _, row in origin_df.iterrows()
    for dest in D
}

In [6]:
Supply = {str(row['City']).strip(): float(row['Capacity']) for _, row in origin_df.iterrows()}
Demand = {dest: float(demand_row[dest]) for dest in D}

Extracting origins and destinations from the spreadsheet.

Adding arcs between all origins and destinations.

In [7]:
A = [(i, j) for i in O for j in D]

Transportation cost for each arc.

In [8]:
c = {
    (str(row['City']).strip(), dest): float(row[dest])
    for _, row in origin_df.iterrows()
    for dest in D
}

Extracting supply and demand values from the spreadsheet.

In [9]:
Supply = {str(row['City']).strip(): float(row['Capacity']) for _, row in origin_df.iterrows()}
Demand = {dest: float(demand_row[dest]) for dest in D}

Making sure that Total Supply >= Total Demand.

In [10]:
if sum(Supply.values()) < sum(Demand.values()):
    diff = sum(Demand.values()) - sum(Supply.values())
    Supply[O[-1]] += diff

###4. Model Preparation

Model Initialization

In [11]:
model = gp.Model('Transportation')

Restricted license - for non-production use only - expires 2027-11-29


Adding decision variables

In [12]:
x = model.addVars(O, D, vtype=GRB.CONTINUOUS, name="x")

Adding supply constraints

In [13]:
for i in O:
    model.addConstr(
        quicksum(x[i, j] for j in D) <= Supply[i],
        name=f"Supply_{i}"
    )

Adding demand constraints

In [14]:
if abs(sum(Supply.values()) - sum(Demand.values())) < 1e-6:
    for j in D:
        model.addConstr(
            quicksum(x[i, j] for i in O) == Demand[j],
            name=f"Demand_{j}"
        )
else:
    # Fallback for unbalanced data
    for j in D:
        model.addConstr(
            quicksum(x[i, j] for i in O) >= Demand[j],
            name=f"Demand_{j}"
        )

Adding the objective function:

In [15]:
model.setObjective(
    quicksum(c[i, j] * x[i, j] for i in O for j in D),
    GRB.MINIMIZE
)

Update and write the model

In [16]:
model.update()
model.write('Transportation.lp')

###5. Model Solution
Solving the model and printing the solution

In [17]:
start = time.time()
model.optimize()
end = time.time()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 13 rows, 40 columns and 80 nonzeros (Min)
Model fingerprint: 0xf3b1aa7e
Model has 40 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+01, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 9e+02]

Presolve time: 0.01s
Presolved: 13 rows, 40 columns, 80 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.7103300e+05   6.320000e+02   0.000000e+00      0s
       6    1.9796100e+05   0.000000e+00   0.000000e+00      0s

Solved in 6 iterations and 0.03 seconds (0.00 work units)
Optimal objective  1.979610000e+05


In [18]:
def print_solution(model):
    if model.status == GRB.OPTIMAL:
        print("\nOptimal Solution Found")
        print("=" * 50)
        print(f"Objective Function Value (Min Cost): {model.objVal:,.2f}")
        print(f"Run Time: {end - start:.4f} seconds")
        print("=" * 50)

        print("\nPositive Shipments:")
        for v in model.getVars():
            if v.X > 1e-6:
                print(f"{v.VarName} = {v.X:,.2f}")
    else:
        print("No optimal solution found.")
        print(f"Model Status Code: {model.status}")

In [19]:
print_solution(model)


Optimal Solution Found
Objective Function Value (Min Cost): 197,961.00
Run Time: 0.0439 seconds

Positive Shipments:
x[Halifax,Saint John] = 187.00
x[London,Moncton] = 345.00
x[London,Ottawa] = 207.00
x[London,Saint John] = 341.00
x[Ajax,Ottawa] = 201.00
x[Ajax,Truro] = 287.00
x[Toronto,Saint John] = 4.00
x[Toronto,Hamilton] = 765.00
x[Montreal,Sydney] = 398.00
x[Montreal,Ottawa] = 54.00
x[Montreal,PEI] = 196.00
x[Montreal,Waterloo] = 15.00
